# Acquisition / provenance

Pass date: **2026-09-05**. Raw files stay unchanged.

Questions this notebook answers:

1. Which file is on disk, and where?
2. When did it get here (`mtime` on this machine)?
3. How big is it (bytes, rows, columns, SHA-256)?
4. Does it have the expected grain and key (`Facility ID` on CMS, `encounter_id` on UCI)?

Write-up: [`data/reference/provenance_phase1.md`](../data/reference/provenance_phase1.md).  
Machine table: [`data/reference/provenance_phase1.csv`](../data/reference/provenance_phase1.csv).  
Checklist: [`docs/source-manifest.md`](../docs/source-manifest.md).

In [ ]:
import csv
import hashlib
from datetime import datetime
from pathlib import Path

ROOT = Path("..").resolve() if Path("../docs/source-manifest.md").exists() else Path(".").resolve()
RAW_CMS = ROOT / "data" / "raw" / "cms"
RAW_UCI = ROOT / "data" / "raw" / "uci"
OUT = ROOT / "data" / "reference" / "provenance_phase1.csv"

EXPECTED = [
    ("CMS", RAW_CMS / "Hospital_General_Information.csv", "facility", "Facility ID"),
    ("CMS", RAW_CMS / "Unplanned_Hospital_Visits-Hospital.csv", "facility x measure", "Facility ID"),
    ("CMS", RAW_CMS / "HCAHPS-Hospital.csv", "facility x survey item", "Facility ID"),
    ("CMS", RAW_CMS / "Timely_and_Effective_Care-Hospital.csv", "facility x measure", "Facility ID"),
    ("CMS", RAW_CMS / "Footnote_Crosswalk.csv", "footnote code", "Footnote"),
    ("CMS", RAW_CMS / "Measure_Dates.csv", "measure", "Measure ID"),
    ("UCI", RAW_UCI / "diabetic_data.csv", "encounter", "encounter_id"),
    ("UCI", RAW_UCI / "IDS_mapping.csv", "stacked ID lookups", None),
]

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

rows = []
missing = []
for layer, path, grain, key in EXPECTED:
    if not path.exists():
        missing.append(path.name)
        print("MISSING", path)
        continue
    with path.open(encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        n = 0
        seen = set()
        ki = header.index(key) if key in header else None
        for rec in reader:
            n += 1
            if ki is not None and ki < len(rec):
                seen.add(rec[ki])
    key_ok = (key is None) or (key in header)
    rec = {
        "file": path.name,
        "path": path.relative_to(ROOT).as_posix(),
        "layer": layer,
        "bytes": path.stat().st_size,
        "mtime_local": datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds"),
        "n_rows": n,
        "n_cols": len(header),
        "grain": grain,
        "key": key or "",
        "n_unique_key": len(seen) if key else "",
        "key_present": key_ok,
        "sha256": sha256(path),
    }
    rows.append(rec)
    print(f"{path.name}: {n:,} rows, {len(header)} cols, key={key} present={key_ok}, unique={rec['n_unique_key']}")

print()
print("ROOT", ROOT)
print("missing", missing or "none")

In [ ]:
OUT.parent.mkdir(parents=True, exist_ok=True)
fields = [
    "file", "path", "layer", "bytes", "mtime_local", "n_rows", "n_cols",
    "grain", "key", "n_unique_key", "sha256",
]
with OUT.open("w", encoding="utf-8", newline="") as f:
    w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
    w.writeheader()
    w.writerows(rows)
print("wrote", OUT)

## Pass notes (2026-09-05)

All eight required files were on disk. I did not clean anything.

- Four hospital CSVs landed 2026-08-29. Footnote crosswalk and measure dates landed 2026-09-05. UCI files have 2023 timestamps (older copy).
- General Information: 5,419 hospitals, one row each, `Facility ID` unique.
- Unplanned Visits / HCAHPS / Timely: long format. Fewer unique facilities than General Information (4,658–4,790). Expected.
- Unplanned measure IDs include the `READM_30_*` and `EDAC_30_*` lines we want. `Hybrid_HWR` is in the file; do not build the story on it.
- UCI: 101,766 unique `encounter_id`, 71,518 patients. `readmitted` is `NO` / `>30` / `<30` (11,357 `<30`).
- `IDS_mapping.csv` is three stacked lookups. Leave it as shipped.

Optional PDF dictionary is in `data/reference/HOSPITAL_Data_Dictionary.pdf`.